# 加载向量数据库

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

from llms.langchain_tutorial.deepseek import get_local_embeddings
from langchain_community.vectorstores import FAISS

# Load the document, split it into chunks, embed each chunk and load it into the vector store.
raw_documents = TextLoader('./examples/rag.txt').load()
text_splitter = CharacterTextSplitter(separator='\n\n\n', chunk_size=50, chunk_overlap=4)
documents = text_splitter.split_documents(raw_documents)
db = FAISS.from_documents(documents, get_local_embeddings())

# LCEL

In [ ]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})

retriever.invoke("哪里可以了解高考成绩")

# MultiQueryRetriever

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from llms.langchain_tutorial.deepseek import get_deepseek_chat

llm = get_deepseek_chat()

retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=retriever, llm=llm, include_original=True
)

unique_docs = retriever_from_llm.invoke("哪里可以了解高考成绩")
unique_docs

In [ ]:
len(unique_docs)

# Contextual compression
## LLMChainExtractor

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    "哪里可以了解高考成绩"
)
compressed_docs

## LLMChainFilter

In [ ]:
from langchain.retrievers.document_compressors import LLMChainFilter

_filter = LLMChainFilter.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=_filter, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    "哪里可以了解高考成绩"
)
compressed_docs

## EmbeddingsFilter

In [ ]:
from langchain.retrievers.document_compressors import EmbeddingsFilter

embeddings = get_local_embeddings()
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.76)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    "What did the president say about Ketanji Jackson Brown"
)
compressed_docs

## DocumentCompressorPipeline

In [ ]:
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter

redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)
relevant_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.76)
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[redundant_filter, relevant_filter]
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(
    "What did the president say about Ketanji Jackson Brown"
)

# Ensemble Retriever

In [ ]:
from langchain_community.retrievers import BM25Retriever

doc_list = [doc.page_content for doc in documents]
bm25_retriever = BM25Retriever.from_texts(
    doc_list, metadatas=[{"source": f"BM25"}] * len(doc_list)
)
bm25_retriever.k = 2

In [ ]:
from langchain.retrievers import EnsembleRetriever

# initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever], weights=[0.5, 0.5]
)

docs = ensemble_retriever.invoke("哪里可以了解高考成绩")
docs

# MultiVector Retriever
## Smaller chunks

In [ ]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryByteStore

import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc_ids = [str(uuid.uuid4()) for _ in documents]
id_key = "doc_id"

# The splitter to use to create smaller chunks
child_text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
sub_docs = []
for i, doc in enumerate(documents):
    _id = doc_ids[i]
    _sub_docs = child_text_splitter.split_documents([doc])
    for _doc in _sub_docs:
        _doc.metadata[id_key] = _id
    sub_docs.extend(_sub_docs)

len(documents), len(sub_docs)

In [ ]:
# The vectorstore to use to index the child chunks
vectorstore = FAISS.from_documents(sub_docs, get_local_embeddings())
# The storage layer for the parent documents
store = InMemoryByteStore()
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)
retriever.docstore.mset(list(zip(doc_ids, documents)))

In [ ]:
# Vectorstore alone retrieves the small chunks
retriever.vectorstore.similarity_search("哪里可以了解高考成绩")[0]

In [ ]:
# Retriever returns larger chunks
retriever.invoke("哪里可以了解高考成绩")[0]

In [ ]:
sub_docs[:5]

## Parent Document Retriever

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain_chroma import Chroma
from langchain.storage import InMemoryStore

# This text splitter is used to create the child documents
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
# The vectorstore to use to index the child chunks (empty to start)
# FAISS not supports empty initialzation
vectorstore = Chroma(
    collection_name="full_documents", embedding_function=get_local_embeddings()
)
# The storage layer for the parent documents
store = InMemoryStore()
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

retriever.add_documents(documents, ids=None)

In [ ]:
sub_docs = vectorstore.similarity_search("哪里可以了解高考成绩")
sub_docs[0]

In [ ]:
retrieved_docs = retriever.invoke("哪里可以了解高考成绩")
retrieved_docs[0]

## Summary

In [ ]:
import uuid

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from llms.langchain_tutorial.deepseek import get_deepseek_chat

chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("概括以下内容:\n\n{doc}")
    | get_deepseek_chat(max_retries=0)
    | StrOutputParser()
)

summaries = chain.batch(documents, {"max_concurrency": 5})
summaries[:2]

In [ ]:
doc_ids = [str(uuid.uuid4()) for _ in documents]
id_key = "doc_id"

summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

# The vectorstore to use to index the child chunks
vectorstore = FAISS.from_documents(summary_docs, get_local_embeddings())
# The storage layer for the parent documents
store = InMemoryByteStore()
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)
retriever.docstore.mset(list(zip(doc_ids, documents)))

In [ ]:
sub_docs = vectorstore.similarity_search("哪里可以了解高考成绩")
sub_docs[0]

In [ ]:
retrieved_docs = retriever.invoke("哪里可以了解高考成绩")
retrieved_docs[0]

## Hypothetical Queries

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.exceptions import OutputParserException

def custom_parse(ai_message: AIMessage) -> str:
    """Parse the AI message."""
    if '\n\n' in ai_message.content:
        return ai_message.content.split('\n\n')
    elif '\n' in ai_message.content:
        return ai_message.content.split('\n')
    else:
        raise OutputParserException("Badly formed question!")

chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("为下面内容生成3个合适的提问问题：\n\n{doc}\n\n#限制\n生成的3个问题使用两个换行符，即```\n\n```符号进行隔开")
    | get_deepseek_chat(max_retries=0)
    | custom_parse
)

hypothetical_questions = chain.batch(documents, {"max_concurrency": 5})
hypothetical_questions[0]

In [ ]:
doc_ids = [str(uuid.uuid4()) for _ in documents]
id_key = "doc_id"

question_docs = []
for i, question_list in enumerate(hypothetical_questions):
    question_docs.extend(
        [Document(page_content=s, metadata={id_key: doc_ids[i]}) for s in question_list]
    )

# The vectorstore to use to index the child chunks
vectorstore = FAISS.from_documents(question_docs, get_local_embeddings())
# The storage layer for the parent documents
store = InMemoryByteStore()
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)
retriever.docstore.mset(list(zip(doc_ids, documents)))

In [ ]:
sub_docs = vectorstore.similarity_search("哪里可以了解高考成绩")
sub_docs[0]

In [ ]:
retrieved_docs = retriever.invoke("哪里可以了解高考成绩")
retrieved_docs[0]

## Self-querying

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]
vectorstore = Chroma.from_documents(docs, get_local_embeddings())

In [ ]:
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever

metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating", description="A 1-10 rating for the movie", type="float"
    ),
]
document_content_description = "Brief summary of a movie"
llm = get_deepseek_chat()
retriever = SelfQueryRetriever.from_llm(
    llm,
    vectorstore,
    document_content_description,
    metadata_field_info,
)

In [ ]:
# This example only specifies a filter
retriever.invoke("I want to watch a movie rated higher than 8.5")
"""
[Document(page_content='Three men walk into the Zone, three men walk out of the Zone', metadata={'director': 'Andrei Tarkovsky', 'genre': 'thriller', 'rating': 9.9, 'year': 1979}),
 Document(page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea', metadata={'director': 'Satoshi Kon', 'rating': 8.6, 'year': 2006})]
"""

### 原理

In [ ]:
from langchain.chains.query_constructor.base import (
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)
from langchain.retrievers.self_query.chroma import ChromaTranslator
from llms.langchain_tutorial.deepseek import get_deepseek_chat

llm = get_deepseek_chat()

prompt = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
    allowed_comparators=ChromaTranslator.allowed_comparators,
    allowed_operators=ChromaTranslator.allowed_operators
)
output_parser = StructuredQueryOutputParser.from_components()
query_constructor = prompt | llm | output_parser

In [ ]:
ChromaTranslator.allowed_comparators

In [ ]:
ChromaTranslator.allowed_operators

In [ ]:
prompt.format(query="dummy question")

In [ ]:
query_constructor.invoke("I want to watch a movie about dinosaurs rated higher than 8.5")

# 2024高考RAG应用
## without rag

In [ ]:
from llms.langchain_tutorial.deepseek import get_deepseek_chat

chat = get_deepseek_chat()

chat.invoke('2024年高考报名人数是多少')

In [ ]:
chat.invoke('2024年高考，广东的报名时间是什么时候')

## with rag

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

from llms.langchain_tutorial.deepseek import get_local_embeddings
from langchain_community.vectorstores import FAISS

# Load the document, split it into chunks, embed each chunk and load it into the vector store.
raw_documents = TextLoader('./examples/rag.txt').load()
text_splitter = CharacterTextSplitter(separator='\n\n\n', chunk_size=50, chunk_overlap=4)
documents = text_splitter.split_documents(raw_documents)
db = FAISS.from_documents(documents, get_local_embeddings())

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage
from langchain_core.exceptions import OutputParserException

def custom_parse(ai_message: AIMessage) -> str:
    """Parse the AI message."""
    if '\n\n' in ai_message.content:
        return ai_message.content.split('\n\n')
    elif '\n' in ai_message.content:
        return ai_message.content.split('\n')
    else:
        raise OutputParserException("Badly formed question!")

chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("为下面内容生成3个合适的提问问题：\n\n{doc}\n\n#限制\n生成的3个问题使用两个换行符，即```\n\n```符号进行隔开")
    | get_deepseek_chat(max_retries=0)
    | custom_parse
)

hypothetical_questions = chain.batch(documents, {"max_concurrency": 5})
hypothetical_questions[0]

In [ ]:
import uuid
from langchain_core.documents import Document
from langchain.storage import InMemoryByteStore
from langchain.retrievers.multi_vector import MultiVectorRetriever

doc_ids = [str(uuid.uuid4()) for _ in documents]
id_key = "doc_id"

question_docs = []
for i, question_list in enumerate(hypothetical_questions):
    question_docs.extend(
        [Document(page_content=s, metadata={id_key: doc_ids[i]}) for s in question_list]
    )

# The vectorstore to use to index the child chunks
vectorstore = FAISS.from_documents(question_docs, get_local_embeddings())
# The storage layer for the parent documents
store = InMemoryByteStore()
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)
retriever.docstore.mset(list(zip(doc_ids, documents)))

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """回答用户的问题，下面的内容可以作为你的知识依据：
```
{context}
```

用户的问题：{query}
"""
prompt = ChatPromptTemplate.from_template(template)


def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])


rag_chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | prompt
    | chat
    | StrOutputParser()
)

rag_chain.invoke("2024年高考报名人数是多少")

In [ ]:
rag_chain.invoke("2024年高考，广东的报名时间是什么时候")

In [ ]:
print(rag_chain.invoke('请概括2024年高考报名人数及变化'))